# Matched-residual deferred-denoising explorer

Scratch space for the idea: instead of letting NORDIC mutate the data and hide the
DoF cost, carry the NORDIC **residual** (the removed thermal noise) through the
*same* preprocessing as the data. You then have, in final processed space:

- **`D'`** = processed *noisy* data  (`A·S + A·N`)
- **`R'`** = processed NORDIC residual = the *matched* thermal noise  (`A·N`, same realization)
- **numcomps'** = the NORDIC components-removed map, interpolated through the same pipeline
- **mask** = brain mask

This notebook examines what `R'` buys you, **assuming NORDIC worked perfectly** so `R'`
is pure thermal noise:

1. the noise covariance `Σ_N` from `R'` (and its interpolation-induced temporal correlation),
2. a whole-brain **signal dimension** = #(directions where `var(D') > var(R')`) — i.e. a
   defensible ICA model order, measured against the real noise floor rather than an MP/parametric null,
3. the **per-patch** signal dimension vs. what NORDIC actually removed — where deferring would
   keep more DoF/signal (or where NORDIC was conservative).

It runs on synthetic data as-is; flip `USE_SYNTHETIC = False` and set the four paths to use yours.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from scipy.linalg import eigh

try:
    import seaborn as sns

    sns.set_style("whitegrid")
except ImportError:
    plt.style.use("seaborn-v0_8-whitegrid") if "seaborn-v0_8-whitegrid" in plt.style.available else None

sys.path.insert(0, str(Path.cwd().parent))

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(0)

## Configuration

Point these at your files (final processed space, all co-registered/same grid), or leave
`USE_SYNTHETIC = True` to demo on a phantom.

In [ ]:
# ---- your data, or leave USE_SYNTHETIC=True ----
USE_SYNTHETIC = True

D_PRIME_PATH  = None   # processed NOISY data D'        (X, Y, Z, T)
R_PRIME_PATH  = None   # processed NORDIC residual R'   (X, Y, Z, T)  -- the matched noise
NUMCOMPS_PATH = None   # interpolated numcomps map      (X, Y, Z)
MASK_PATH     = None   # brain mask                     (X, Y, Z)

KERNEL       = (7, 7, 7)  # patch size for the per-patch analysis
PATCH_STRIDE = 2          # >1 = sparser patch grid (faster while exploring)
EIG_MARGIN   = 0.0        # count generalized eigenvalues > 1 + EIG_MARGIN as signal

In [ ]:
def _load(p):
    return np.asarray(nib.load(str(p)).get_fdata(), dtype=np.float32)


if USE_SYNTHETIC:
    # Phantom: a few shared low-rank "signal" timecourses with focal spatial
    # loadings + i.i.d. thermal noise. R' is exactly that noise (perfect NORDIC).
    X, Y, Z, T = 24, 24, 8, 120
    nvox = X * Y * Z
    n_sig = 6
    Vt = np.linalg.qr(rng.standard_normal((T, n_sig)))[0]              # (T, n_sig)
    U = rng.standard_normal((nvox, n_sig)) * (rng.random((nvox, n_sig)) < 0.10)  # focal
    amp = np.array([8, 6, 5, 4, 3, 2.5], dtype=np.float32)
    signal = (U * amp) @ Vt.T                                          # (nvox, T)
    noise = rng.standard_normal((nvox, T)).astype(np.float32)
    Dp = (signal + noise).reshape(X, Y, Z, T)
    Rp = noise.reshape(X, Y, Z, T)                                     # matched residual
    mask = np.ones((X, Y, Z), dtype=bool)
    mask[0] = mask[-1] = False                                         # trim x edges
    # Plausible NORDIC numcomps: keeps a bit more than the true signal dim where
    # signal is strong (so kept ~ 9..18 of T), i.e. removes most of the T dims.
    sig_energy = (signal ** 2).sum(1).reshape(X, Y, Z)
    keep_target = n_sig + 3 + 6 * sig_energy / (sig_energy.max() + 1e-9)
    numcomps = np.clip(T - keep_target, 5, T - 2).astype(np.float32)
else:
    Dp = _load(D_PRIME_PATH)
    Rp = _load(R_PRIME_PATH)
    numcomps = _load(NUMCOMPS_PATH)
    mask = _load(MASK_PATH) > 0

assert Dp.shape == Rp.shape, (Dp.shape, Rp.shape)
assert mask.shape == Dp.shape[:3]
X, Y, Z, T = Dp.shape
print(f"D'/R' shape: {Dp.shape}   mask voxels: {int(mask.sum())}   T = {T}")

## Sanity: tSNR of D', the noise level of R', and the numcomps map

In [ ]:
def tsnr(vol):
    m = vol.mean(-1)
    s = vol.std(-1)
    out = np.zeros_like(m)
    nz = s > 0
    out[nz] = m[nz] / s[nz]
    return out


zc = Z // 2
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
im0 = ax[0].imshow(tsnr(Dp)[:, :, zc].T, origin="lower")
ax[0].set_title("tSNR  D' (signal+noise)")
plt.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(Rp.std(-1)[:, :, zc].T, origin="lower")
ax[1].set_title("R' temporal std (matched noise)")
plt.colorbar(im1, ax=ax[1], fraction=0.046)
im2 = ax[2].imshow(numcomps[:, :, zc].T, origin="lower")
ax[2].set_title("numcomps removed (NORDIC)")
plt.colorbar(im2, ax=ax[2], fraction=0.046)
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def masked_TV(vol, m):
    """(V, T): voxel time series inside mask, mean-removed over time."""
    v = vol[m]
    return v - v.mean(1, keepdims=True)


def temporal_cov(M):
    """T x T temporal covariance, voxels as samples (well-conditioned when V > T)."""
    return (M.T @ M) / (M.shape[0] - 1)


def shrink(C, n_samples):
    """Ledoit-Wolf-ish shrink toward mu*I. Regularizes the rank-1 deficiency that
    temporal demeaning leaves in C (the mean direction) and the M<=T case."""
    p = C.shape[0]
    a = float(min(0.5, p / max(1, n_samples)))
    mu = np.trace(C) / p
    return (1 - a) * C + a * mu * np.eye(p)


Dm = masked_TV(Dp, mask)
Rm = masked_TV(Rp, mask)
V = Rm.shape[0]
print(f"masked: D' {Dm.shape}, R' {Rm.shape}  (need V > T = {T} for a stable T x T cov: {V > T})")

## The noise covariance from R'

`Σ_N = R'ᵀR' / (V−1)` is the temporal noise covariance. Off-diagonal structure is the
**interpolation-induced temporal correlation** — if the pipeline left the noise white, `Σ_N`
would be diagonal. This is the object NORDIC's MP threshold tries to *predict*; here it's *measured*.

In [ ]:
Sigma_N = temporal_cov(Rm)
Sigma_D = temporal_cov(Dm)
evN = np.linalg.eigvalsh(Sigma_N)[::-1]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
vmax = np.abs(Sigma_N).max()
im = ax[0].imshow(Sigma_N, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax[0].set_title(r"Noise temporal covariance $\Sigma_N$ (from R')")
plt.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].plot(evN, marker=".")
ax[1].set_title(r"$\Sigma_N$ eigenspectrum")
ax[1].set_xlabel("index")
ax[1].set_ylabel("variance")
plt.tight_layout()
plt.show()
# Drop the smallest eigenvalue: temporal demeaning leaves a rank-1 (mean-direction)
# null in Sigma_N, so the raw condition number is meaningless. Use the rest.
cond = evN[0] / max(evN[-2], 1e-12)
print(f"Sigma_N condition number (mean direction excluded): {cond:.2f}  "
      f"(near 1 = still ~white; large = interpolation correlated the noise)")

In [ ]:
# Noise temporal autocorrelation implied by Sigma_N (mean over its diagonals).
d = np.sqrt(np.diag(Sigma_N))
corrN = Sigma_N / np.outer(d, d)
lags = np.arange(min(30, T))
acf = np.array([np.mean(np.diagonal(corrN, k)) for k in lags])
plt.figure(figsize=(8, 4))
plt.stem(lags, acf)
plt.axhline(0, color="k", lw=0.8)
plt.title("Noise temporal autocorrelation from $\Sigma_N$  (nonzero lags = interpolation correlation)")
plt.xlabel("lag (TR)")
plt.ylabel("mean corr")
plt.tight_layout()
plt.show()

## Whole-brain signal dimension

Generalized eigenproblem `Σ_D v = λ Σ_N v`: each `λ` is the ratio of data variance to noise
variance along a temporal direction. Naively you'd count `λ > 1`, but that **over-counts** — under
the null (data = matched noise) the `λ` don't sit at 1, they spread (a Wachter/MANOVA distribution),
so ~half land above 1 by chance. The fix is the whole point of having `R'`: **measure the null
directly** by splitting `R'` in half and whitening one noise half by the other. The signal
dimension is the count of `λ_data` above the upper edge of that empirical null — no MP/parametric
assumption.

In [ ]:
# Empirical null: split R' voxels in half; the generalized eigenvalues of one
# noise half whitened by the other ARE the H0 distribution (data == matched noise).
perm = rng.permutation(V)
half = V // 2
Ra, Rb = Rm[perm[:half]], Rm[perm[half:]]
Sn = shrink(temporal_cov(Rb), Rb.shape[0])
lam_null = eigh(temporal_cov(Ra), Sn, eigvals_only=True)[::-1]
lam_data = eigh(Sigma_D, Sn, eigvals_only=True)[::-1]
NULL_EDGE = float(np.percentile(lam_null, 99))
N_signal = int((lam_data > NULL_EDGE).sum())
evD = np.linalg.eigvalsh(Sigma_D)[::-1]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].axhline(NULL_EDGE, color="r", ls="--", lw=1, label=f"null 99th pct = {NULL_EDGE:.2f}")
ax[0].plot(lam_data, marker=".", label="D' vs R'")
ax[0].plot(lam_null, marker=".", ms=3, alpha=0.5, label="null (R' vs R')")
ax[0].set_yscale("log")
ax[0].set_title(f"Whitened spectrum   ->   N_signal = {N_signal}")
ax[0].set_xlabel("component")
ax[0].set_ylabel(r"variance ratio $\lambda$")
ax[0].legend()
ax[1].plot(evD / evD.sum(), marker=".")
ax[1].set_yscale("log")
ax[1].set_title("Naive PCA variance fraction (no noise reference)")
ax[1].set_xlabel("component")
plt.tight_layout()
plt.show()
print(f"Whole-brain signal dimension (lambda above the empirical null edge): {N_signal}")
print(f"  -> use as the whole-brain ICA model order.  null edge = {NULL_EDGE:.3f}")

## Per-patch: signal dimension vs. what NORDIC removed

Same generalized-eigenvalue count, per patch, thresholded at the **whole-brain empirical null
edge** (`NULL_EDGE`), compared to NORDIC's kept dimension `T − numcomps`. The delta tells you where
deferring would differ:

- **delta > 0** (signal dim > NORDIC kept): NORDIC removed *more* than the noise floor justifies
  → over-removal, the deferred scheme would keep those DoF/signal.
- **delta < 0**: NORDIC was conservative there.

Caveat: the *true* per-patch null edge depends on the patch voxel count `M`, so reusing the global
edge is an approximation — fine for eyeballing, worth a per-scale null if you take this further.
`PATCH_STRIDE` subsamples the grid; patches with fewer voxels than timepoints are skipped (the
`T×T` covariance needs `M > T`).

In [ ]:
kx, ky, kz = KERNEL
step = (max(1, kx // 2) * PATCH_STRIDE, max(1, ky // 2) * PATCH_STRIDE, max(1, kz // 2) * PATCH_STRIDE)
xs = list(range(0, max(1, X - kx + 1), step[0]))
ys = list(range(0, max(1, Y - ky + 1), step[1]))
zs = list(range(0, max(1, Z - kz + 1), step[2]))

centers, n_sig_patch, nordic_removed_patch = [], [], []
for x0 in xs:
    for y0 in ys:
        for z0 in zs:
            sub = (slice(x0, x0 + kx), slice(y0, y0 + ky), slice(z0, z0 + kz))
            msub = mask[sub]
            if int(msub.sum()) <= int(1.2 * T):   # need M > T for a stable T x T cov
                continue
            dp = Dp[sub][msub]
            rp = Rp[sub][msub]
            dp = dp - dp.mean(1, keepdims=True)
            rp = rp - rp.mean(1, keepdims=True)
            Snp = shrink(temporal_cov(rp), rp.shape[0])
            Sdp = temporal_cov(dp)
            lp = eigh(Sdp, Snp, eigvals_only=True)
            centers.append((x0 + kx // 2, y0 + ky // 2, z0 + kz // 2))
            n_sig_patch.append(int((lp > NULL_EDGE + EIG_MARGIN).sum()))
            nordic_removed_patch.append(float(numcomps[sub][msub].mean()))

centers = np.array(centers)
n_sig_patch = np.array(n_sig_patch, dtype=float)
nordic_removed_patch = np.array(nordic_removed_patch)
nordic_kept = T - nordic_removed_patch
delta = n_sig_patch - nordic_kept   # >0 -> deferred keeps more dims than NORDIC did

print(f"patches analyzed: {len(centers)}")
if len(centers):
    print(f"median R'-whitened signal dim : {np.median(n_sig_patch):.1f}")
    print(f"median NORDIC kept (T-numcomps): {np.median(nordic_kept):.1f}")
    print(f"median dim delta               : {np.median(delta):+.1f}")

In [ ]:
if len(centers):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    sc = ax[0].scatter(nordic_kept, n_sig_patch, c=delta, cmap="coolwarm", s=18,
                       vmin=-np.abs(delta).max() - 1e-6, vmax=np.abs(delta).max() + 1e-6)
    lim = [0, max(nordic_kept.max(), n_sig_patch.max()) + 2]
    ax[0].plot(lim, lim, "k--", lw=1)
    ax[0].set_xlim(lim)
    ax[0].set_ylim(lim)
    ax[0].set_xlabel("NORDIC kept (T - numcomps)")
    ax[0].set_ylabel("R'-whitened signal dim")
    ax[0].set_title("per patch: signal dim vs NORDIC kept")
    plt.colorbar(sc, ax=ax[0], label="dim delta")
    ax[1].hist(delta, bins=30)
    ax[1].axvline(0, color="k", ls="--")
    ax[1].set_title("dim delta = signal_dim - NORDIC_kept")
    ax[1].set_xlabel("per patch")
    ax[2].hist(n_sig_patch, bins=30, alpha=0.6, label="R'-whitened signal dim")
    ax[2].hist(nordic_kept, bins=30, alpha=0.6, label="NORDIC kept")
    ax[2].legend()
    ax[2].set_title("kept / signal dim distributions")
    plt.tight_layout()
    plt.show()

In [ ]:
if len(centers):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    s0 = ax[0].scatter(centers[:, 0], centers[:, 1], c=n_sig_patch, cmap="viridis", s=28)
    ax[0].set_title("signal dim per patch (XY projection)")
    plt.colorbar(s0, ax=ax[0])
    s1 = ax[1].scatter(centers[:, 0], centers[:, 1], c=delta, cmap="coolwarm", s=28,
                       vmin=-np.abs(delta).max() - 1e-6, vmax=np.abs(delta).max() + 1e-6)
    ax[1].set_title("dim delta per patch (XY projection)")
    plt.colorbar(s1, ax=ax[1])
    for a in ax:
        a.set_aspect("equal")
        a.set_xlabel("x")
        a.set_ylabel("y")
    plt.tight_layout()
    plt.show()

## Scratch / things to poke at

- **The whitening assumes `R'` is pure noise.** If NORDIC over-removed (the cross-echo QC
  hotspots), `R'` carries that signal → it inflates `Σ_N` exactly where signal is, biasing
  `λ` *down* there. Cross-check the dim-delta map against `_resid_xcorr` / `_recfactor`.
- **`Σ_N` whole-brain is fine (V ≫ T); per-patch needs M > T** — raise `KERNEL` if patches get
  skipped, or shrink `Σ_N` (Ledoit–Wolf) instead of the ridge.
- **The generalized eigenvalue is variance-ratio, not SNR per voxel** — a direction with `λ`
  just above 1 is barely-detectable signal; try `EIG_MARGIN > 0` to be stricter.
- **Next experiments:** (a) does `D' − R'` reproduce the denoise-then-process data (plumbing/
  linearity check)? (b) how many `R'` components recover NORDIC's variance reduction? (c) hold-out
  R² at the QC-flagged over-removal voxels — where deferred should win.
- Whitened `D'` (the `Σ_N^{-1/2}` rotation) is also the right input for ICA and for an empirical
  prewhitening covariance; and running your stats on `R'` *alone* is a real-data null for
  cluster/FPR calibration.